# Program *Training*: Jaringan Saraf Tiruan (*Backpropagation*)

Program ini melatih komputer untuk mengenali 3 jenis kendaraan (**pesawat, mobil, dan kapal**) dari gambar hitam-putih (*grayscale*) berukuran 32 x 32 piksel.

Metode yang digunakan adalah *Backpropagation* standar (Fausett, 1994). Hasil akhirnya adalah satu berkas bobot `BOBOT.npz` yang siap digunakan untuk mengenali gambar baru.

## Sel 1. Pengaturan Awal dan Konfigurasi Jaringan

* **Penyimpanan:** Menghubungkan Google Drive untuk membaca folder gambar (latih dan uji) serta menentukan tempat menyimpan hasil latihan (`BOBOT.npz`).
* **Bentuk Jaringan:**
  * **Input (1024 node):** Berasal dari seluruh piksel gambar ($32 \times 32$).
  * **Hidden (100 node):** Lapisan perantara untuk menangkap ciri khas gambar.
  * **Output (3 node):** Lapisan penentu tebakan (pesawat, mobil, atau kapal).
* **Aturan Latihan:**
  * `alpha = 0.1` (*learning rate*): Kecepatan penyesuaian bobot tiap kali model salah tebak.
  * `EPOCH_MAKS = 200`: Batas maksimal perulangan belajar seluruh dataset.
  * `AMBANG_GALAT = 0.01`: Target tingkat kesalahan minimal agar latihan bisa berhenti lebih cepat.

> **Analogi:** Menyiapkan ruang kelas: menentukan materi soal yang akan dipelajari, seberapa tegas koreksi guru jika murid salah, dan batas waktu latihan.

In [ ]:
import os                      # akses folder dan berkas
import time                    # mengukur lama training
import numpy as np             # operasi matriks dan vektor
from PIL import Image          # membaca gambar (Pillow)
import matplotlib.pyplot as plt  # menggambar grafik dan gambar
from google.colab import drive   # menyambungkan Colab ke Google Drive

drive.mount('/content/drive')  # sambungkan Colab ke Google Drive

# Lokasi di Google Drive
FOLDER_PROYEK = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI'
FOLDER_LATIH  = FOLDER_PROYEK + '/DATASET/LATIH'   # gambar untuk belajar
FOLDER_UJI    = FOLDER_PROYEK + '/DATASET/UJI'     # gambar untuk menguji (tidak ikut dilatih)
BERKAS_BOBOT  = FOLDER_PROYEK + '/BOBOT/BOBOT.npz' # tempat menyimpan hasil training

# Ukuran gambar dan arsitektur
LEBAR, TINGGI = 32, 32  # ukuran gambar dalam pixel (titik gambar)
n = LEBAR * TINGGI      # node input: satu node untuk tiap pixel
p = 100                 # node hidden: lapisan tengah pengolah pola
m = 3                   # node output: satu per kelas (harus tepat 3)

# Parameter pembelajaran
alpha        = 0.1      # learning rate: besar langkah koreksi bobot tiap pembaruan
EPOCH_MAKS   = 200      # epoch: satu putaran penuh seluruh data latih; ini batas maksimum (syarat berhenti 1)
AMBANG_GALAT = 0.01     # galat: selisih keluaran dan target; berhenti bila rata-rata per pola di bawah ini (syarat berhenti 2)
BATAS_ACAK   = 0.5      # bobot awal diacak antara -0,5 dan 0,5
BIJI_ACAK    = 42       # random seed: angka awal pengacak agar hasil bisa diulang
CETAK_TIAP   = 10       # interval laporan: cetak hasil tiap 10 epoch

# Nama folder tiap kelas. Urutannya menentukan target: indeks 0 = 1 0 0, indeks 1 = 0 1 0, indeks 2 = 0 0 1
NAMA_KELAS = ['KELAS1_PESAWAT', 'KELAS2_MOBIL', 'KELAS3_KAPAL']

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Node input  (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p) : {p}')
print(f'Node output (m) : {m}')
print(f'alpha           : {alpha}')
print(f'Epoch maksimum  : {EPOCH_MAKS}')
print(f'Ambang galat    : {AMBANG_GALAT}')

## Sel 2. Penyiapan Gambar dan Memuat *Dataset*

* **Fungsi `praproses`:** Mengubah gambar menjadi format angka siap latih melalui 4 tahap:
  1. *Grayscale:* Mengubah gambar berwarna menjadi abu-abu.
  2. *Resize:* Menyeragamkan ukuran menjadi $32 \times 32$ piksel.
  3. *Flatten:* Merentangkan gambar 2 dimensi menjadi 1 deretan panjang berisi 1024 angka.
  4. *Normalisasi:* Membagi nilai piksel dengan 255 agar nilainya berada di skala 0 sampai 1.
* **Fungsi `muat_dataset`:** Membaca semua gambar serta menetapkan kunci jawaban (*one-hot target*):
  * Pesawat: `1 0 0`
  * Mobil: `0 1 0`
  * Kapal: `0 0 1`

> **Analogi:** Menyalin semua foto soal menjadi format hitam-putih seragam seukuran kartu pos, lalu mencatat kunci jawabannya.

In [ ]:
def praproses(gambar):
    """Ubah satu gambar PIL (RGB) menjadi vektor x berisi n nilai, rentang 0 sampai 1."""
    abu = gambar.convert('L')                       # 1. grayscale: warna jadi satu nilai terang per pixel
    if abu.size != (LEBAR, TINGGI):                 # 2. pengaman ukuran: bila ukuran beda,
        abu = abu.resize((LEBAR, TINGGI))           #    ubah ukuran (resize) ke 32 x 32
    x = np.asarray(abu, dtype=np.float64).flatten() # 3. flatten: ubah gambar 2 dimensi jadi satu baris
    return x / 255.0                                # 4. normalisasi: bagi 255 agar nilai 0 sampai 1


def muat_dataset(folder):
    """Baca semua gambar di folder. Hasil: X (data), T (target), label (nomor kelas), nama berkas."""
    X, T, label, nama_berkas = [], [], [], []
    for k, nama in enumerate(NAMA_KELAS):
        # sorted: urutkan nama berkas agar urutan baca selalu sama
        daftar = sorted(f for f in os.listdir(f'{folder}/{nama}') if f.endswith('.png'))
        for berkas in daftar:
            gambar = Image.open(f'{folder}/{nama}/{berkas}')
            X.append(praproses(gambar))
            target = np.zeros(m)
            target[k] = 1.0                         # one-hot: hanya node kelas ini bernilai 1 (100, 010, atau 001)
            T.append(target)
            label.append(k)                         # label: nomor kelas (0, 1, atau 2), dipakai menghitung akurasi
            nama_berkas.append(f'{nama}/{berkas}')
    return np.array(X), np.array(T), np.array(label), nama_berkas


X_latih, T_latih, label_latih, _ = muat_dataset(FOLDER_LATIH)
X_uji,   T_uji,   label_uji,   _ = muat_dataset(FOLDER_UJI)

print('=' * 52)
print('DATASET DIMUAT')
print('=' * 52)
print(f'{"Kelas":<18}{"Latih":>8}{"Uji":>8}   Target')
for k, nama in enumerate(NAMA_KELAS):
    target = ' '.join(str(int(a)) for a in np.eye(m)[k])
    print(f'{nama:<18}{(label_latih == k).sum():>8}{(label_uji == k).sum():>8}   {target}')
print('-' * 52)
print(f'{"Total":<18}{len(X_latih):>8}{len(X_uji):>8}')
print(f'\nBentuk X_latih : {X_latih.shape}  (pola x node input)')
print(f'Rentang nilai  : {X_latih.min():.2f} sampai {X_latih.max():.2f}')

## Sel 3. Cek Visual Gambar Hasil Konversi

* Membandingkan gambar asli berwarna (baris atas) dengan hasil olahan *grayscale* (baris bawah) untuk memastikan bentuk objek kendaraan tetap utuh dan jelas sebelum masuk proses latihan.

> **Analogi:** Memeriksa fotokopi soal sebelum dibagikan untuk memastikan gambar tidak rusak atau buram.

In [ ]:
# baris atas: gambar asli berwarna; baris bawah: hasil praproses (grayscale)
fig, sumbu = plt.subplots(2, 3, figsize=(7, 5))
for k, nama in enumerate(NAMA_KELAS):
    # ambil gambar pertama tiap kelas sebagai contoh
    berkas = sorted(f for f in os.listdir(f'{FOLDER_LATIH}/{nama}') if f.endswith('.png'))[0]
    asli = Image.open(f'{FOLDER_LATIH}/{nama}/{berkas}')
    sumbu[0, k].imshow(asli)
    sumbu[0, k].set_title(nama.split('_')[1] + f'\n(mode {asli.mode})')
    sumbu[1, k].imshow(praproses(asli).reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1)
    sumbu[1, k].set_title('grayscale')
    sumbu[0, k].axis('off'); sumbu[1, k].axis('off')
plt.tight_layout()
plt.show()

## Sel 4. Rumus Aktivasi dan Inisialisasi Bobot Awal (Langkah 0)

* **Fungsi Aktivasi Sigmoid:** Mengompres hasil perhitungan jaringan agar selalu berada dalam skala 0 sampai 1, beserta rumus turunannya untuk menghitung koreksi galat.
* **Bobot Awal:** Nilai sambungan antarlapisan (`v`, `w`) dan *bias* (`v_0`, `w_0`) diisi angka acak antara -0.5 sampai 0.5 sebagai tebakan awal model.

> **Analogi:** Bobot acak seperti seorang murid yang menebak jawaban secara asal-asalan sebelum mulai diajari materi.

In [ ]:
def f(x):
    # fungsi aktivasi sigmoid: ubah nilai apa pun menjadi 0 sampai 1
    return 1.0 / (1.0 + np.exp(-x))

def f_aksen(x):
    # turunan sigmoid: dipakai menghitung koreksi galat
    return f(x) * (1.0 - f(x))


# Langkah 0: inisialisasi bobot
rng = np.random.default_rng(BIJI_ACAK)               # rng: pembangkit bilangan acak (memakai random seed di atas)
v   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (n, p))   # bobot V: dari node input ke node hidden
v_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, p)        # bias node hidden: nilai tambahan tetap tiap node hidden
w   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (p, m))   # bobot W: dari node hidden ke node output
w_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, m)        # bias node output: nilai tambahan tetap tiap node output

print('=' * 52)
print('BOBOT AWAL (Langkah 0)')
print('=' * 52)
print(f'{"Variabel":<10}{"Ukuran":<14}{"Jumlah nilai":>12}')
for nama, larik in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)]:
    print(f'{nama:<10}{str(larik.shape):<14}{larik.size:>12,}')
print(f'\nRentang bobot awal: {min(v.min(), w.min()):.2f} sampai {max(v.max(), w.max()):.2f}')

## Sel 5. Siklus Pelatihan *Backpropagation* (Langkah 1 sampai 9)

Latihan berjalan per putaran (*epoch*) melalui alur berikut:
1. **Maju (*Feedforward*):** Gambar dihitung dari lapisan *input* $\rightarrow$ *hidden* $\rightarrow$ keluar berupa tebakan di lapisan *output* (Langkah 3 sampai 5).
2. **Hitung Galat:** Menghitung selisih antara tebakan jaringan dengan kunci jawaban (target).
3. **Mundur (*Backpropagation*):** Menyalurkan nilai kesalahan ke belakang untuk memperbaiki bobot dan *bias* (Langkah 6 sampai 8).
4. **Penghentian:** Proses berulang sampai galat di bawah 0.01 atau telah mencapai 200 *epoch* (Langkah 9).

> **Analogi:** Murid mengerjakan soal (maju), melihat kuncinya lalu mengetahui di mana salahnya (galat), lalu memperbaiki cara berpikirnya agar soal berikutnya tidak salah lagi (mundur).

In [ ]:
def umpan_maju(X):
    """Umpan maju (feedforward): hitung keluaran y dari X. Hanya untuk melaporkan kemajuan, tidak mengubah bobot."""
    z = f(v_0 + X @ v)           # keluaran node hidden
    return f(w_0 + z @ w)        # keluaran node output

def akurasi(X, label):
    # argmax: nomor node output bernilai terbesar = kelas tebakan; akurasi = persen tebakan yang benar
    return np.mean(np.argmax(umpan_maju(X), axis=1) == label) * 100


riwayat_galat, riwayat_akurasi = [], []   # catatan galat dan akurasi tiap epoch, untuk grafik
jumlah_pola = len(X_latih)                # pola = satu gambar latih beserta targetnya

print('=' * 52)
print('TRAINING')
print('=' * 52)
print(f'{"Epoch":>6}{"Galat rata-rata":>18}{"Akurasi latih":>16}')
print('-' * 52)

waktu_mulai = time.time()

# Langkah 1: ulangi selama syarat berhenti belum terpenuhi (satu putaran = satu epoch)
for epoch in range(1, EPOCH_MAKS + 1):
    galat_total = 0.0                      # jumlah galat semua pola dalam epoch ini

    # Langkah 2: untuk setiap pasangan (x, t); urutan pola diacak tiap epoch (rng.permutation)
    for s in rng.permutation(jumlah_pola):
        x = X_latih[s]                     # x: masukan (nilai 1024 pixel)
        t = T_latih[s]                     # t: target (100, 010, atau 001)

        # Langkah 3 sampai 5: umpan maju (feedforward)
        z_in = v_0 + x @ v                 # z_in: masukan bersih node hidden
        z    = f(z_in)                     # z: keluaran node hidden
        y_in = w_0 + z @ w                 # y_in: masukan bersih node output
        y    = f(y_in)                     # y: keluaran node output

        # Langkah 6: galat di node output (delta_k: faktor koreksi galat)
        delta_k   = (t - y) * f_aksen(y_in)
        delta_w   = alpha * np.outer(z, delta_k)   # np.outer: perkalian luar, hasil matriks ukuran p x m
        delta_w_0 = alpha * delta_k                # koreksi bias node output

        # Langkah 7: galat di node hidden (memakai w lama)
        delta_in_j = w @ delta_k           # jumlah galat output yang dikirim balik ke node hidden
        delta_j    = delta_in_j * f_aksen(z_in)
        delta_v    = alpha * np.outer(x, delta_j)  # np.outer: hasil matriks ukuran n x p
        delta_v_0  = alpha * delta_j               # koreksi bias node hidden

        # Langkah 8: perbarui bobot dan bias (bobot baru = bobot lama + koreksi)
        w   = w   + delta_w
        w_0 = w_0 + delta_w_0
        v   = v   + delta_v
        v_0 = v_0 + delta_v_0

        galat_total += 0.5 * np.sum((t - y) ** 2)  # galat satu pola: setengah jumlah kuadrat selisih target dan keluaran

    galat_rata = galat_total / jumlah_pola         # galat rata-rata per pola pada epoch ini
    riwayat_galat.append(galat_rata)
    riwayat_akurasi.append(akurasi(X_latih, label_latih))

    if epoch == 1:
        print(f'(waktu 1 epoch: {time.time() - waktu_mulai:.1f} detik)')
    if epoch == 1 or epoch % CETAK_TIAP == 0:
        print(f'{epoch:>6}{galat_rata:>18.5f}{riwayat_akurasi[-1]:>15.1f}%')

    # Langkah 9: uji syarat berhenti
    if galat_rata < AMBANG_GALAT:
        print(f'\nBerhenti: galat rata-rata di bawah ambang pada epoch {epoch}.')
        break
else:
    # else pada for: dijalankan bila loop selesai tanpa break
    print(f'\nBerhenti: epoch maksimum ({EPOCH_MAKS}) tercapai.')

print('-' * 52)
print(f'Epoch dijalankan : {len(riwayat_galat)}')
print(f'Galat akhir      : {riwayat_galat[-1]:.5f}')
print(f'Akurasi latih    : {riwayat_akurasi[-1]:.1f}%')
print(f'Waktu total      : {time.time() - waktu_mulai:.1f} detik')

## Sel 6. Grafik Hasil Evaluasi Belajar

* Menampilkan grafik perubahan kesalahan (*galat*) dan akurasi sepanjang proses latihan.
* Pelatihan dinilai berhasil jika kurva galat terus menurun mendekati nol dan kurva akurasi terus menanjak mendekati 100%.

> **Analogi:** Grafik rapor kemajuan belajar murid: apakah kesalahan ujian berkurang dan nilainya makin naik dari waktu ke waktu.

In [ ]:
# kiri: galat tiap epoch (diharapkan turun); kanan: akurasi data latih (diharapkan naik)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(range(1, len(riwayat_galat) + 1), riwayat_galat)
a1.set_title('Galat rata-rata per epoch'); a1.set_xlabel('Epoch'); a1.set_ylabel('Galat'); a1.grid(alpha=0.3)
a2.plot(range(1, len(riwayat_akurasi) + 1), riwayat_akurasi, color='tab:green')
a2.set_title('Akurasi data latih (%)'); a2.set_xlabel('Epoch'); a2.set_ylim(0, 100); a2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Sel 7. Evaluasi Model dan *Confusion Matrix*

* **Uji Kemampuan:** Membandingkan akurasi data latih dengan data uji untuk memeriksa gejala *overfitting* (kondisi saat model hanya hafal data latih tetapi gagal menebak gambar baru).
* **Confusion Matrix:** Tabel detail tebakan model:
  * Angka diagonal: Tebakan yang benar.
  * Angka di luar diagonal: Jumlah kesalahan tebak antarkelas (misal: kapal yang salah dikenali sebagai pesawat).

> **Analogi:** Ujian akhir memakai soal baru. Jika latihan dapat nilai 100 tapi di soal baru nilainya anjlok, berarti murid tersebut sekadar menghafal soal latihan (*overfitting*).

In [ ]:
def laporan(X, label, judul):
    tebakan = np.argmax(umpan_maju(X), axis=1)   # tebakan: nomor node output bernilai terbesar
    benar = np.mean(tebakan == label) * 100
    print(f'{judul}: {benar:.1f}%  ({int((tebakan == label).sum())} dari {len(label)} gambar benar)')
    return tebakan

print('=' * 52)
print('EVALUASI')
print('=' * 52)
laporan(X_latih, label_latih, 'Akurasi data latih')
tebakan_uji = laporan(X_uji, label_uji, 'Akurasi data uji  ')

# confusion matrix: baris = kelas sebenarnya, kolom = tebakan model
print('\nTabel tebakan pada data uji')
judul = [nama.split('_')[1] for nama in NAMA_KELAS]
print(f'{"Sebenarnya":<12}' + ''.join(f'{j:>9}' for j in judul) + f'{"Akurasi":>10}')
for k in range(m):
    baris = [int(np.sum((label_uji == k) & (tebakan_uji == j))) for j in range(m)]
    persen = baris[k] / np.sum(label_uji == k) * 100
    print(f'{judul[k]:<12}' + ''.join(f'{b:>9}' for b in baris) + f'{persen:>9.0f}%')

## Sel 8. Detail Nilai *Output* vs Kunci Jawaban

* Menampilkan nilai desimal pada node output ($Y_1, Y_2, Y_3$) pada sampel data uji untuk memeriksa ketegasan model.
* Model dinyatakan yakin jika node kelas yang benar bernilai mendekati 1 dan node lainnya mendekati 0.

> **Analogi:** Memeriksa lembar jawaban pilihan ganda: melihat seberapa yakin murid memilih jawabannya, bukan sekadar ragu-ragu di antara dua pilihan.

In [ ]:
print('=' * 62)
print('POLA TARGET OUTPUT (t)')
print('=' * 62)
print(f'{"Kelas":<18}{"Y1":>4}{"Y2":>4}{"Y3":>4}   Artinya')
for k in range(m):
    pola = ''.join(f'{int(a):>4}' for a in np.eye(m)[k])
    print(f'{NAMA_KELAS[k]:<18}{pola}   hanya Y{k + 1} bernilai 1')

print()
print('=' * 62)
print('KELUARAN JARINGAN (y) DIBANDINGKAN DENGAN TARGET (t), DATA UJI')
print('=' * 62)
print(f'{"Kelas":<16}{"Target t":<10}{"Keluaran y (Y1, Y2, Y3)":<26}{"Hasil"}')
print('-' * 62)
y_uji = umpan_maju(X_uji)   # keluaran jaringan (y) untuk seluruh data uji
for k in range(m):
    for s in np.where(label_uji == k)[0][:3]:
        t_teks = ' '.join(str(int(a)) for a in T_uji[s])
        y_teks = '  '.join(f'{a:.3f}' for a in y_uji[s])
        # BENAR bila node dengan nilai terbesar adalah node kelas sebenarnya
        hasil = 'BENAR' if np.argmax(y_uji[s]) == k else 'SALAH'
        print(f'{NAMA_KELAS[k].split('_')[1]:<16}{t_teks:<10}{y_teks:<26}{hasil}')

## Sel 9. Menyimpan Bobot Terlatih

* Menyimpan parameter bobot final (`v`, `v_0`, `w`, `w_0`) dan informasi ukuran gambar ke dalam berkas `BOBOT.npz` di Google Drive.
* Berkas ini nantinya langsung dimuat oleh program pengujian (*testing/inference*) tanpa perlu melatih jaringan dari awal lagi.

> **Analogi:** Menyimpan catatan rumus dan intisari pelajaran ke dalam satu buku saku agar siap dipakai kapan saja tanpa harus mengulang sekolah dari nol.

In [ ]:
os.makedirs(os.path.dirname(BERKAS_BOBOT), exist_ok=True)   # buat folder BOBOT bila belum ada
# savez: simpan beberapa larik NumPy dalam satu berkas .npz
# ukuran gambar ikut disimpan agar program pengujian memakai ukuran yang sama
np.savez(BERKAS_BOBOT, v=v, v_0=v_0, w=w, w_0=w_0, ukuran=np.array([LEBAR, TINGGI]))

# Pemeriksaan: baca ulang berkas yang baru disimpan
cek = np.load(BERKAS_BOBOT)
print('=' * 52)
print('BOBOT TERSIMPAN')
print('=' * 52)
print(f'Lokasi : {BERKAS_BOBOT}')
print(f'Ukuran : {os.path.getsize(BERKAS_BOBOT) / 1024:.0f} KB\n')
print(f'{"Isi":<10}{"Ukuran":<14}')
for kunci in ['v', 'v_0', 'w', 'w_0']:
    print(f'{kunci:<10}{str(cek[kunci].shape):<14}')
sama = all(np.array_equal(cek[k], a) for k, a in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)])
print(f'\nIsi berkas sama dengan bobot di memori: {"YA" if sama else "TIDAK"}')